<a href="https://colab.research.google.com/github/emanhassan2020/HandsOn/blob/main/LLM/florence_images_summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers timm einops accelerate pillow

In [2]:
!pip install transformers==4.49.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the

microsoft's florence-2 colab caption sequence of images ang generate story and summary

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
start_path = '/content/drive/MyDrive/returning_back/TestData/Images/'

def list_image_files(directory):
    image_files = []
    image_extensions = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp')
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(image_extensions):
                image_files.append(os.path.join(root, file))
    return image_files

# Get the list of image files
my_images = list_image_files(start_path)

# Display the list of image files found
print(f"Found {len(my_images)} image files:")
for img_file in my_images:
    print(img_file)

Found 13 image files:
/content/drive/MyDrive/returning_back/TestData/Images/img1.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img2.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img3.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img4.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img5.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img6.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img7.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img8.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img9.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img10.jpeg
/content/drive/MyDrive/returning_back/TestData/Images/img11.jpg
/content/drive/MyDrive/returning_back/TestData/Images/img12.jpg
/content/drive/MyDrive/returning_back/TestData/Images/img13.jpg


In [5]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Initialize Device and Florence-2 for Image Captioning
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading Microsoft Florence-2-large...")
florence_model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-large",
    trust_remote_code=True,
    torch_dtype=torch_dtype
).to(device).eval()
florence_processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

# 2. Initialize a Lightweight LLM for Storytelling & Summarization
print("Loading SmolLM2 text generator...")
llm_pipeline = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    torch_dtype=torch_dtype,
    device_map="auto"
)

def generate_caption(image_path, prompt_type="<MORE_DETAILED_CAPTION>"):
    """Uses Florence-2 to extract context from a given image."""
    try:
        image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"Error loading image {image_path}: {str(e)}"

    inputs = florence_processor(text=prompt_type, images=image, return_tensors="pt").to(device, torch_dtype)

    with torch.no_grad():
        generated_ids = florence_model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            num_beams=3
        )

    generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    # Florence-2 returns parsed dictionaries matching the task prompt
    parsed_answer = florence_processor.post_process_generation(
        generated_text,
        task=prompt_type,
        image_size=(image.width, image.height)
    )
    return parsed_answer[prompt_type]

# 3. Define your sequential pipeline
def pipeline_images_to_story(image_paths):
    # Step A: Gather Captions sequentially
    print("\n--- Phase 1: Image Captioning with Florence-2 ---")
    sequential_captions = []
    for idx, img_path in enumerate(image_paths):
        print(f"Processing image {idx + 1}/{len(image_paths)}: {img_path}")
        caption = generate_caption(img_path, prompt_type="<MORE_DETAILED_CAPTION>")
        print(f"Resulting Caption: {caption}\n")
        sequential_captions.append(f"Scene {idx+1}: {caption}")

    context_string = "\n".join(sequential_captions)

    # Step B: Construct Story Generation Prompt
    story_messages = [
        {"role": "system", "content": "You are a creative writer. Connect the provided chronological image descriptions into a seamless, engaging narrative story."},
        {"role": "user", "content": f"Here is the sequence of events based on the images:\n{context_string}\n\nWrite a coherent story linking these moments together."}
    ]

    story_prompt = llm_pipeline.tokenizer.apply_chat_template(story_messages, tokenize=False, add_generation_prompt=True)
    story_out = llm_pipeline(story_prompt, max_new_tokens=500, do_sample=True, temperature=0.7)
    story_text = story_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1]

    # Step C: Construct Summarization Prompt
    summary_messages = [
        {"role": "system", "content": "You are an expert summarizer. Summarize the overall plot or message of the generated story in 2-3 impact sentences."},
        {"role": "user", "content": f"Story:\n{story_text}"}
    ]

    summary_prompt = llm_pipeline.tokenizer.apply_chat_template(summary_messages, tokenize=False, add_generation_prompt=True)
    summary_out = llm_pipeline(summary_prompt, max_new_tokens=150, do_sample=False)
    summary_text = summary_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1]

    # Display Final Deliverable Output
    print("=" * 50)
    print("📖 GENERATED STORY:")
    print("=" * 50)
    print(story_text.strip())
    print("\n" + "=" * 50)
    print("📌 SUMMARY:")
    print("=" * 50)
    print(summary_text.strip())




Loading Microsoft Florence-2-large...


config.json: 0.00B [00:00, ?B/s]

configuration_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

Device set to use cuda:0


In [6]:
# --- Execution Example ---
# Replace these strings with your actual uploaded file names inside Colab (e.g., 'frame1.jpg')
#my_images = ["image_part1.jpg", "image_part2.jpg", "image_part3.jpg"]
# Get the list of image files
my_images = list_image_files(start_path)
# Uncomment below to run if you have uploaded the mock images
pipeline_images_to_story(my_images)


--- Phase 1: Image Captioning with Florence-2 ---
Processing image 1/13: /content/drive/MyDrive/returning_back/TestData/Images/img1.jpeg
Resulting Caption: The image shows two young children, a boy and a girl, sitting on the floor of a living room. They are both looking at a black camera with a strap attached to it. The girl is wearing a purple dress and has curly hair, while the boy has dark hair. There are several other items scattered around them, including a pink water bottle, a notebook, and some colorful blocks. The floor is covered with a gray and white patterned rug. In the background, there is a white curtain and a glass door.

Processing image 2/13: /content/drive/MyDrive/returning_back/TestData/Images/img2.jpeg
Resulting Caption: The image shows a young boy sitting on the floor in a living room, playing with colorful building blocks. He is wearing a yellow and black striped shirt and is holding a green toy in his hands. The blocks are of different shapes and sizes, includin

In [7]:
"""
Prompt Engineering: The script uses <MORE_DETAILED_CAPTION>.
       If the model is seeing too many small details and missing the big picture context,
       you can change prompt_type in the execution call to <DETAILED_CAPTION> or <CAPTION> for shorter, action-oriented frames.

Text Parsing: The post_process_generation method cleanly handles the Florence-2 native text formatting syntax so your story generation model gets clean data sentences.

"""

'\nPrompt Engineering: The script uses <MORE_DETAILED_CAPTION>.\n       If the model is seeing too many small details and missing the big picture context,\n       you can change prompt_type in the execution call to <DETAILED_CAPTION> or <CAPTION> for shorter, action-oriented frames.\n\nText Parsing: The post_process_generation method cleanly handles the Florence-2 native text formatting syntax so your story generation model gets clean data sentences.\n\n'

customizing the storytelling tone as diary

In [10]:
def pipeline_images_to_diary(image_paths):
    # Step A: Gather Captions sequentially with Florence-2
    print("\n--- Phase 1: Image Captioning with Florence-2 ---")
    sequential_captions = []
    for idx, img_path in enumerate(image_paths):
        print(f"Processing image {idx + 1}/{len(image_paths)}: {img_path}")
        caption = generate_caption(img_path, prompt_type="<MORE_DETAILED_CAPTION>")
        print(f"Resulting Caption: {caption}\n")
        sequential_captions.append(f"Scene {idx+1} observations: {caption}")

    context_string = "\n".join(sequential_captions)

    # Step B: Construct Diary Generation Prompt (Customized Tone)
    diary_messages = [
        {
            "role": "system",
            "content": (
                "You are writing a deeply personal, introspective diary entry. "
                "Transform the provided visual observations into a continuous, first-person narrative ('I'). "
                "Organize the output into entries with mock dates or timestamps (e.g., 'October 14th - Morning'). "
                "Focus on feelings, internal thoughts, and personal reactions to what is seen in the scenes."
            )
        },
        {
            "role": "user",
            "content": f"Here is the timeline of what was witnessed or experienced:\n{context_string}\n\nWrite the diary entries linking these moments."
        }
    ]

    story_prompt = llm_pipeline.tokenizer.apply_chat_template(diary_messages, tokenize=False, add_generation_prompt=True)
    story_out = llm_pipeline(story_prompt, max_new_tokens=600, do_sample=True, temperature=0.75)
    diary_text = story_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "")

    # Step C: Construct Summarization Prompt
    summary_messages = [
        {
            "role": "system",
            "content": "You are a concise summarizer. Summarize the overarching theme and emotional evolution of these diary entries in 2-3 impactful sentences."
        },
        {
            "role": "user",
            "content": f"Diary Entries:\n{diary_text}"
        }
    ]

    summary_prompt = llm_pipeline.tokenizer.apply_chat_template(summary_messages, tokenize=False, add_generation_prompt=True)
    summary_out = llm_pipeline(summary_prompt, max_new_tokens=150, do_sample=False)
    summary_text = summary_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "")

    # Display Final Deliverable Output
    print("=" * 50)
    print("📔 PERSONAL DIARY ENTRIES:")
    print("=" * 50)
    print(diary_text.strip())
    print("\n" + "=" * 50)
    print("📌 EMOTIONAL & PLOT SUMMARY:")
    print("=" * 50)
    print(summary_text.strip())

# --- Execution ---
#my_images = ["image_part1.jpg", "image_part2.jpg", "image_part3.jpg"]
pipeline_images_to_diary(my_images)



--- Phase 1: Image Captioning with Florence-2 ---
Processing image 1/13: /content/drive/MyDrive/returning_back/TestData/Images/img1.jpeg
Resulting Caption: The image shows two young children, a boy and a girl, sitting on the floor of a living room. They are both looking at a black camera with a strap attached to it. The girl is wearing a purple dress and has curly hair, while the boy has dark hair. There are several other items scattered around them, including a pink water bottle, a notebook, and some colorful blocks. The floor is covered with a gray and white patterned rug. In the background, there is a white curtain and a glass door.

Processing image 2/13: /content/drive/MyDrive/returning_back/TestData/Images/img2.jpeg
Resulting Caption: The image shows a young boy sitting on the floor in a living room, playing with colorful building blocks. He is wearing a yellow and black striped shirt and is holding a green toy in his hands. The blocks are of different shapes and sizes, includin

can you customize the code for questions answering about the generated summary and story

In [11]:
def pipeline_images_to_diary_with_qa(image_paths):
    # Step A: Gather Captions sequentially with Florence-2
    print("\n--- Phase 1: Image Captioning with Florence-2 ---")
    sequential_captions = []
    for idx, img_path in enumerate(image_paths):
        caption = generate_caption(img_path, prompt_type="<MORE_DETAILED_CAPTION>")
        sequential_captions.append(f"Scene {idx+1} observations: {caption}")

    context_string = "\n".join(sequential_captions)

    # Step B: Construct Diary Generation Prompt
    diary_messages = [
        {"role": "system", "content": "You are writing a deeply personal diary entry. Transform the visual observations into a continuous, first-person narrative ('I'). Organize into entries with mock date headers."},
        {"role": "user", "content": f"Timeline:\n{context_string}"}
    ]
    story_prompt = llm_pipeline.tokenizer.apply_chat_template(diary_messages, tokenize=False, add_generation_prompt=True)
    story_out = llm_pipeline(story_prompt, max_new_tokens=600, do_sample=True, temperature=0.75)
    diary_text = story_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    # Step C: Construct Summarization Prompt
    summary_messages = [
        {"role": "system", "content": "Summarize the overarching theme and emotional evolution of these diary entries in 2-3 impactful sentences."},
        {"role": "user", "content": f"Diary Entries:\n{diary_text}"}
    ]
    summary_prompt = llm_pipeline.tokenizer.apply_chat_template(summary_messages, tokenize=False, add_generation_prompt=True)
    summary_out = llm_pipeline(summary_prompt, max_new_tokens=150, do_sample=False)
    summary_text = summary_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    # Display Deliverable
    print("=" * 50)
    print("📔 PERSONAL DIARY ENTRIES:")
    print("=" * 50)
    print(diary_text)
    print("\n" + "=" * 50)
    print("📌 EMOTIONAL & PLOT SUMMARY:")
    print("=" * 50)
    print(summary_text)

    # Return context variables for the QA function
    return diary_text, summary_text


def ask_diary_question(diary_text, summary_text, user_question):
    """Answers user queries directly based on the generated narrative and summary."""
    qa_messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question accurately using only the provided "
                "Diary Entries and Summary context. If the answer cannot be found or inferred, say so clearly."
            )
        },
        {
            "role": "user",
            "content": f"Context Document:\n\n--- DIARY ENTRIES ---\n{diary_text}\n\n--- SUMMARY ---\n{summary_text}\n\nQuestion: {user_question}"
        }
    ]

    qa_prompt = llm_pipeline.tokenizer.apply_chat_template(qa_messages, tokenize=False, add_generation_prompt=True)
    qa_out = llm_pipeline(qa_prompt, max_new_tokens=200, do_sample=False)
    answer = qa_out[0]['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    return answer


In [ ]:
# 1. Run pipeline and unpack outputs
#my_images = ["image_part1.jpg", "image_part2.jpg", "image_part3.jpg"]
diary_content, summary_content = pipeline_images_to_diary_with_qa(my_images)

# 2. Interactive QA Terminal Loop
print("\n" + "?" * 50)
print("💬 INTERACTIVE DIARY QA SYSTEM")
print("Type your question below. Type 'exit' to quit.")
print("?" * 50)

while True:
    question = input("\nYour Question: ")
    if question.lower().strip() == 'exit':
        print("Exiting QA session.")
        break
    if not question.strip():
        continue

    print("Thinking...")
    reply = ask_diary_question(diary_content, summary_content, question)
    print(f"Answer: {reply}")



--- Phase 1: Image Captioning with Florence-2 ---
📔 PERSONAL DIARY ENTRIES:
I'm ready to respond.

📌 EMOTIONAL & PLOT SUMMARY:
I'm ready to respond.

??????????????????????????????????????????????????
💬 INTERACTIVE DIARY QA SYSTEM
Type your question below. Type 'exit' to quit.
??????????????????????????????????????????????????

Your Question: what toys the children play with
Thinking...
Answer: The children play with various toys in the diary entries.

Your Question: how old are the children in the diary ?


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Thinking...
Answer: The children in the diary are 10 and 12 years old.

Your Question: how many children are there ?
Thinking...
Answer: There are 2 children in the family.


persistently log the conversation history i

In [ ]:
import os

def interactive_chat_session(diary_text, summary_text, log_filename="diary_qa_transcript.txt"):
    """
    Launches a multi-turn chat loop with memory and exports the history to a file.
    """
    # 1. Initialize the message history with context and system constraints
    chat_history = [
        {
            "role": "system",
            "content": (
                "You are an AI assistant helping the user explore a generated narrative. "
                "Answer questions strictly based on the provided Diary Entries and Summary. "
                "Maintain continuity by remembering previous turns in this conversation."
            )
        },
        {
            "role": "user",
            "content": f"Here is the context for our conversation:\n\n--- DIARY ENTRIES ---\n{diary_text}\n\n--- SUMMARY ---\n{summary_text}\n\nUnderstood? Please acknowledge."
        },
        {
            "role": "assistant",
            "content": "I have successfully analyzed the Diary Entries and Summary context. I am ready to answer your questions and will remember our ongoing discussion. What would you like to know?"
        }
    ]

    print("\n" + "💬" * 25)
    print("  CONVERSATIONAL DIARY QA SYSTEM (WITH MEMORY)")
    print(f"  Transcript will save to: {log_filename}")
    print("  Type 'exit' to quit.")
    print("💬" * 25)
    print(f"\nAssistant: {chat_history[-1]['content']}")

    # 2. Start conversational loop
    while True:
        user_input = input("\nYou: ").strip()

        if user_input.lower() == 'exit':
            print("\nEnding chat session. Writing transcript to disk...")
            break
        if not user_input:
            continue

        # Append current question to history matrix
        chat_history.append({"role": "user", "content": user_input})
        print("Thinking...")

        # Format the entire conversation window for SmolLM2
        qa_prompt = llm_pipeline.tokenizer.apply_chat_template(
            chat_history,
            tokenize=False,
            add_generation_prompt=True
        )

        # Generate answer using current history context
        qa_out = llm_pipeline(qa_prompt, max_new_tokens=250, do_sample=False)
        answer = qa_out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

        # Append answer to hold memory for the next loop cycle
        chat_history.append({"role": "assistant", "content": answer})
        print(f"\nAssistant: {answer}")

    # 3. Export conversation history to file upon exit
    try:
        with open(log_filename, "w", encoding="utf-8") as f:
            f.write("==================================================\n")
            f.write("📜 DIARY NARRATIVE CHAT TRANSCRIPT\n")
            f.write("==================================================\n\n")
            for msg in chat_history:
                role_label = msg['role'].upper()
                # Skip dumping the massive raw diary block inside the log file for readability
                if "--- DIARY ENTRIES ---" in msg['content']:
                    f.write("SYSTEM: [Context Document Transmitted to Model]\n\n")
                else:
                    f.write(f"{role_label}: {msg['content']}\n\n")
        print(f"✅ Success! Transcript saved to your Colab workspace as '{log_filename}'.")
    except Exception as e:
        print(f"❌ Warning: Could not write transcript file: {str(e)}")

# --- Execution Example ---
# Assuming 'diary_content' and 'summary_content' were unpacked from your pipeline run:
# interactive_chat_session(diary_content, summary_content)


support LangGraph state serialization and save to Google Drive folder

In [ ]:
# Install LangGraph
!pip install langgraph transformers timm einops accelerate pillow

# Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Establish project save directory
DRIVE_DIR = "/content/drive/MyDrive/Florence2_LangGraph_Diary"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Project artifacts will sync automatically to: {DRIVE_DIR}")


In [ ]:
import os
import json
import torch
from typing import List, Dict, Any, TypedDict
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, pipeline
from langgraph.graph import StateGraph, END

# ==========================================
# 1. HARDWARE & TRANSFORMS ENGINE INITIALIZATION
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Loading Foundations (Florence-2 & SmolLM2)...")
florence_model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True, torch_dtype=torch_dtype).to(device).eval()
florence_processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

llm_pipeline = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.3B-Instruct", torch_dtype=torch_dtype, device_map="auto")

# ==========================================
# 2. DEFINING LANGGRAPH STATE SCHEMAS
# ==========================================
class DiaryPipelineDict(TypedDict):
    image_paths: List[str]
    raw_captions: List[str]
    diary_narrative: str
    emotional_summary: str
    chat_history: List[Dict[str, str]]
    metadata: Dict[str, Any]

# ==========================================
# 3. PIPELINE GRAPH NODES IMPLEMENTATION
# ==========================================
def caption_extraction_node(state: DiaryPipelineDict) -> Dict[str, Any]:
    """Node 1: Iterates through sequential images using Florence-2."""
    paths = state.get("image_paths", [])
    captions = []

    print(f"\n[Node: Caption Extraction] Processing {len(paths)} images...")
    for idx, img_path in enumerate(paths):
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = florence_processor(text="<MORE_DETAILED_CAPTION>", images=image, return_tensors="pt").to(device, torch_dtype)
            with torch.no_grad():
                generated_ids = florence_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=512, num_beams=3)
            generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=True)
            parsed_answer = florence_processor.post_process_generation(generated_text, task="<MORE_DETAILED_CAPTION>", image_size=(image.width, image.height))
            caption_str = parsed_answer["<MORE_DETAILED_CAPTION>"]
            captions.append(f"Scene {idx+1}: {caption_str}")
            print(f" Successfully processed: {img_path}")
        except Exception as e:
            captions.append(f"Scene {idx+1} Error: Failed to process image {img_path}. Exception: {str(e)}")

    return {"raw_captions": captions}

def diary_generation_node(state: DiaryPipelineDict) -> Dict[str, Any]:
    """Node 2: Compiles the generated scene metadata into a diary layout using SmolLM2."""
    print("\n[Node: Diary Generation] Authoring narrative...")
    context_string = "\n".join(state["raw_captions"])

    messages = [
        {"role": "system", "content": "You are writing a deeply personal diary entry. Transform visual logs into a continuous, first-person narrative ('I'). Organize using timestamped headers."},
        {"role": "user", "content": f"Visual logs timeline:\n{context_string}"}
    ]
    prompt = llm_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(prompt, max_new_tokens=600, do_sample=True, temperature=0.75)
    diary_text = out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    return {"diary_narrative": diary_text}

def summary_generation_node(state: DiaryPipelineDict) -> Dict[str, Any]:
    """Node 3: Formulates the final summary analysis block."""
    print("\n[Node: Summary Generation] Distilling semantic vectors...")
    diary_text = state["diary_narrative"]

    messages = [
        {"role": "system", "content": "Summarize the overarching emotional progression of the text in 2-3 impact sentences."},
        {"role": "user", "content": f"Text:\n{diary_text}"}
    ]
    prompt = llm_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(prompt, max_new_tokens=150, do_sample=False)
    summary_text = out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    # Pre-populate Chat QA base layer messages inside the state buffer
    initial_chat = [
        {"role": "system", "content": "Answer user questions strictly based on the provided Diary Entries and Summary. Maintain conversation history continuity."},
        {"role": "user", "content": f"Context Document:\n\n--- DIARY ENTRIES ---\n{diary_text}\n\n--- SUMMARY ---\n{summary_text}\n\nAcknowledge parsing."},
        {"role": "assistant", "content": "Context parsed. I am ready to process multi-turn QA queries based on this narrative structure."}
    ]

    return {
        "emotional_summary": summary_text,
        "chat_history": initial_chat,
        "metadata": {"execution_status": "COMPLETED", "torch_precision": "float16"}
    }

# ==========================================
# 4. COMPOSING THE STATE GRAPH ENGINE
# ==========================================
workflow = StateGraph(DiaryPipelineDict)

# Register Nodes
workflow.add_node("extract_captions", caption_extraction_node)
workflow.add_node("generate_diary", diary_generation_node)
workflow.add_node("generate_summary", summary_generation_node)

# Set Entry point and linear execution paths
workflow.set_entry_point("extract_captions")
workflow.add_edge("extract_captions", "generate_diary")
workflow.add_edge("generate_diary", "generate_summary")
workflow.add_edge("generate_summary", END)

# Compile Graph
app = workflow.compile()

# ==========================================
# 5. SERIALIZATION UTILITIES (Drive Synchronization)
# ==========================================
def serialize_and_save_state(state: DiaryPipelineDict, target_dir: str, run_id: str = "latest_run"):
    """
    Serializes the LangGraph pipeline state dictionary to JSON format
    and commits all generated textual output assets straight into Google Drive.
    """
    state_filename = os.path.join(target_dir, f"{run_id}_state_checkpoint.json")
    transcript_filename = os.path.join(target_dir, f"{run_id}_readable_transcript.txt")

    # Save serialized JSON state checkpoint
    try:
        with open(state_filename, "w", encoding="utf-8") as json_file:
            json.dump(state, json_file, indent=4, ensure_ascii=False)
        print(f"\n💾 [Serialized Checkpoint] State committed successfully to: {state_filename}")
    except Exception as e:
        print(f"⚠️ Serialization execution failed: {str(e)}")

    # Save a clean, human-readable workspace report text document
    try:
        with open(transcript_filename, "w", encoding="utf-8") as txt_file:
            txt_file.write("==================================================\n")
            txt_file.write(f"📝 LANGGRAPH DIARY NARRATIVE SYSTEM TRANSCRIPT | RUN: {run_id}\n")
            txt_file.write("==================================================\n\n")
            txt_file.write("📔 GENERATED DIARY:\n")
            txt_file.write(f"{state.get('diary_narrative', '')}\n\n")
            txt_file.write("📌 EMOTIONAL & PLOT SUMMARY:\n")
            txt_file.write(f"{state.get('emotional_summary', '')}\n\n")
            txt_file.write("🤖 CAPTION OBSERVATIONS OBTAINED:\n")
            txt_file.write("\n".join(state.get('raw_captions', [])))
        print(f"📄 [Human Readable Document] Transcript compiled at: {transcript_filename}")
    except Exception as e:
        print(f"⚠️ Narrative report generation failed: {str(e)}")


In [ ]:
# 1. Define input image sequence parameters (Ensure these exist in your Colab runtime)
initial_input = {
    "image_paths": ["image_part1.jpg", "image_part2.jpg", "image_part3.jpg"],
    "raw_captions": [],
    "diary_narrative": "",
    "emotional_summary": "",
    "chat_history": [],
    "metadata": {}
}

# 2. Invoke Graph Execution
print("Executing LangGraph Linear Node Chain...")
final_output_state = app.invoke(initial_input)

# 3. Serialize initial state snapshot to Google Drive
serialize_and_save_state(final_output_state, target_dir=DRIVE_DIR, run_id="session_01")

# ==========================================
# 6. STATEFUL INTERACTIVE CHAT LOOP WITH LIVE DRIVE SYNC
# ==========================================
print("\n" + "💬" * 25)
print("  LANGGRAPH STATEFUL MULTI-TURN QA SYSTEM")
print("  Every interaction syncs state parameters back to Drive.")
print("  Type 'exit' to gracefully close out the session.")
print("💬" * 25)

history_buffer = final_output_state["chat_history"]

while True:
    user_query = input("\nYou: ").strip()
    if user_query.lower() == 'exit':
        print("\nSession complete. Synchronizing final graph checkpoints to Drive...")
        break
    if not user_query:
        continue

    # Append message tracking to state framework array
    history_buffer.append({"role": "user", "content": user_query})
    print("Thinking...")

    # Structure full prompt payload matrices matching context constraints
    qa_prompt = llm_pipeline.tokenizer.apply_chat_template(history_buffer, tokenize=False, add_generation_prompt=True)
    qa_out = llm_pipeline(qa_prompt, max_new_tokens=300, do_sample=False)
    assistant_reply = qa_out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    # Complete state tracking sequence
    history_buffer.append({"role": "assistant", "content": assistant_reply})
    print(f"\nAssistant: {assistant_reply}")

    # Overwrite running dictionary values & flash update states straight into Google Drive files
    final_output_state["chat_history"] = history_buffer
    serialize_and_save_state(final_output_state, target_dir=DRIVE_DIR, run_id="session_01")


can you update the system to answer questions about something in the images then retrieve the corresponding related images from the input image sequence

In [ ]:
import os
import json
import torch
from typing import List, Dict, Any, TypedDict
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM, pipeline
from langgraph.graph import StateGraph, END

# ==========================================
# 1. HARDWARE & ENGINE INITIALIZATION
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Initializing Models (Florence-2 & SmolLM2)...")
florence_model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True, torch_dtype=torch_dtype).to(device).eval()
florence_processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)
llm_pipeline = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.3B-Instruct", torch_dtype=torch_dtype, device_map="auto")

# ==========================================
# 2. DEFINING UPDATED NARRATIVE STATE SCHEMA
# ==========================================
class VisualDiaryState(TypedDict):
    image_paths: List[str]
    # Maps specific scene indices to their file path and descriptive captions
    scene_mappings: List[Dict[str, str]]
    diary_narrative: str
    emotional_summary: str
    chat_history: List[Dict[str, str]]
    metadata: Dict[str, Any]

# ==========================================
# 3. UPDATED PIPELINE GRAPH NODES
# ==========================================
def extraction_and_mapping_node(state: VisualDiaryState) -> Dict[str, Any]:
    """Extracts captions and maps them clearly to their originating file paths."""
    paths = state.get("image_paths", [])
    mappings = []

    print(f"\n[Node: Mapping Extraction] Processing {len(paths)} images...")
    for idx, img_path in enumerate(paths):
        try:
            image = Image.open(img_path).convert("RGB")
            inputs = florence_processor(text="<MORE_DETAILED_CAPTION>", images=image, return_tensors="pt").to(device, torch_dtype)
            with torch.no_grad():
                generated_ids = florence_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=512, num_beams=3)
            generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=True)
            parsed_answer = florence_processor.post_process_generation(generated_text, task="<MORE_DETAILED_CAPTION>", image_size=(image.width, image.height))
            caption_str = parsed_answer["<MORE_DETAILED_CAPTION>"]

            # Save explicit file tracking alongside captions
            mappings.append({
                "scene_id": f"Scene {idx+1}",
                "file_path": img_path,
                "caption": caption_str
            })
            print(f" Successfully mapped: {img_path}")
        except Exception as e:
            mappings.append({"scene_id": f"Scene {idx+1}", "file_path": img_path, "caption": f"Error parsing: {str(e)}"})

    return {"scene_mappings": mappings}

def narrative_generation_node(state: VisualDiaryState) -> Dict[str, Any]:
    """Generates the diary entries from the structured caption mappings."""
    print("\n[Node: Diary Generation] Composing narrative flow...")
    context_list = [f"{m['scene_id']}: {m['caption']}" for m in state["scene_mappings"]]
    context_string = "\n".join(context_list)

    messages = [
        {"role": "system", "content": "You are writing a personal diary entry. Transform visual logs into a continuous, first-person narrative ('I'). Organize using timestamped headers."},
        {"role": "user", "content": f"Timeline logs:\n{context_string}"}
    ]
    prompt = llm_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(prompt, max_new_tokens=600, do_sample=True, temperature=0.75)
    diary_text = out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    return {"diary_narrative": diary_text}

def execution_summary_node(state: VisualDiaryState) -> Dict[str, Any]:
    """Summarizes text and configures foundational conversational prompt windows."""
    diary_text = state["diary_narrative"]

    messages = [
        {"role": "system", "content": "Summarize the overarching emotional progression of the text in 2-3 sentences."},
        {"role": "user", "content": f"Text:\n{diary_text}"}
    ]
    prompt = llm_pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(prompt, max_new_tokens=150, do_sample=False)
    summary_text = out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    initial_chat = [
        {"role": "system", "content": "Answer user questions based on the provided Diary and Summary. If they ask about objects or things in the images, answer normally based on the text context. The secondary router system will isolate target image frames."},
        {"role": "user", "content": f"Context Documents:\n\n--- DIARY ENTRIES ---\n{diary_text}\n\n--- SUMMARY ---\n{summary_text}"},
        {"role": "assistant", "content": "Context parsed. I am ready to process conversation flows."}
    ]

    return {"emotional_summary": summary_text, "chat_history": initial_chat, "metadata": {"status": "ACTIVE"}}

# ==========================================
# 4. COMPILING LANGGRAPH ARCHITECTURE
# ==========================================
workflow = StateGraph(VisualDiaryState)
workflow.add_node("extract_mappings", extraction_and_mapping_node)
workflow.add_node("generate_diary", narrative_generation_node)
workflow.add_node("generate_summary", execution_summary_node)

workflow.set_entry_point("extract_mappings")
workflow.add_edge("extract_mappings", "generate_diary")
workflow.add_edge("generate_diary", "generate_summary")
workflow.add_edge("generate_summary", END)
app = workflow.compile()

# ==========================================
# 5. VISUAL CAPTION RETRIEVAL ENGINE
# ==========================================
def find_relevant_images(user_query: str, mappings: List[Dict[str, str]]) -> List[str]:
    """
    Scans the user query for noun objects and searches for matching
    visual items within the sequence using Florence-2's grounding engine.
    """
    matched_image_paths = []

    # We ask the smaller, smart LLM to distill the exact core noun phrase from the query
    distill_messages = [
        {"role": "system", "content": "Extract only the main visual target object/noun being asked about from the user query. Output just the object name and nothing else (e.g., 'cat', 'red car', 'laptop')."},
        {"role": "user", "content": f"Query: {user_query}"}
    ]
    prompt = llm_pipeline.tokenizer.apply_chat_template(distill_messages, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(prompt, max_new_tokens=20, do_sample=False)
    target_object = out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip().lower()

    print(f"🔍 [Retrieval Engine] Searching for keyword reference: '{target_object}'...")

    for item in mappings:
        img_path = item["file_path"]
        try:
            image = Image.open(img_path).convert("RGB")
            # Using phrase grounding to check if the extracted item physically exists in the image context
            grounding_prompt = f"<CAPTION_TO_PHRASE_GROUNDING>{target_object}"
            inputs = florence_processor(text=grounding_prompt, images=image, return_tensors="pt").to(device, torch_dtype)

            with torch.no_grad():
                generated_ids = florence_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], max_new_tokens=256, num_beams=3)

            generated_text = florence_processor.batch_decode(generated_ids, skip_special_tokens=True)
            parsed_answer = florence_processor.post_process_generation(generated_text, task="<CAPTION_TO_PHRASE_GROUNDING>", image_size=(image.width, image.height))

            # If coordinates / bounding boxes are successfully returned, the object is present
            grounding_results = parsed_answer["<CAPTION_TO_PHRASE_GROUNDING>"]
            if grounding_results.get("labels"):
                matched_image_paths.append(img_path)
        except Exception:
            # Fallback to string matching over raw captions if grounding fails
            if target_object in item["caption"].lower():
                matched_image_paths.append(img_path)

    return list(set(matched_image_paths))


In [ ]:
# 1. Provide your images
initial_inputs = {
    "image_paths": ["image_part1.jpg", "image_part2.jpg", "image_part3.jpg"],
    "scene_mappings": [], "diary_narrative": "", "emotional_summary": "", "chat_history": [], "metadata": {}
}

# Execute LangGraph Sequence
state_result = app.invoke(initial_inputs)
history = state_result["chat_history"]

# 2. Start Conversation Loop
print("\n" + "💬" * 25)
print("  LANGGRAPH IMAGE-AWARE CHAT & RETRIEVAL SYSTEM")
print("  Type 'exit' to conclude.")
print("💬" * 25)

while True:
    query = input("\nYou: ").strip()
    if query.lower() == 'exit':
        break
    if not query:
        continue

    history.append({"role": "user", "content": query})
    print("Thinking...")

    # Textual Generation Response
    prompt = llm_pipeline.tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
    out = llm_pipeline(prompt, max_new_tokens=300, do_sample=False)
    reply = out['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    history.append({"role": "assistant", "content": reply})
    print(f"\nAssistant: {reply}")

    # Secondary Step: Trigger Image Sequence Look-up Engine
    matched_files = find_relevant_images(query, state_result["scene_mappings"])
    if matched_files:
        print("\n🖼️ [RETRIEVED RELATED SOURCE IMAGES]:")
        for f in matched_files:
            print(f" -> {f}")
    else:
        print("\n🖼️ [No matching item localized directly inside image frames]")


conditional router edge to skip the LLM evaluation step if the vector database confidence scores are too low

┌───────────────────────┐
                  │   extract_mappings    │
                  └───────────┬───────────┘
                              │
                              ▼
                  ┌───────────────────────┐
                  │    evaluate_vector    │
                  └───────────┬───────────┘
                              │
                     [Conditional Router]
                    /                    \
     (Confidence >= 0.7)              (Confidence < 0.7)
                  /                        \
                 ▼                          ▼
     ┌───────────────────────┐  ┌───────────────────────┐
     │    generate_diary     │  │  low_confidence_node  │
     └───────────┬───────────┘  └───────────┬───────────┘
                 │                          │
                 ▼                          ▼
     ┌───────────────────────┐              │
     │   generate_summary    │              │
     └───────────┬───────────┘              │
                 │                          │
                 └────────────┬─────────────┘
                              │
                              ▼
                           [ END ]

In [ ]:
import os
import torch
from typing import List, Dict, Any, TypedDict, Literal
from PIL import Image
from langgraph.graph import StateGraph, END

# ==========================================
# 1. EXTENDED DIARY NARRATIVE STATE SCHEMA
# ==========================================
class RoutedDiaryState(TypedDict):
    image_paths: List[str]
    scene_mappings: List[Dict[str, str]]
    diary_narrative: str
    emotional_summary: str
    chat_history: List[Dict[str, str]]
    metadata: Dict[str, Any]
    # New routing metric parameter trackers
    vector_confidence_score: float
    routing_decision: str

# ==========================================
# 2. CORE WORKFLOW PIPELINE NODES
# ==========================================
def extraction_and_mapping_node(state: RoutedDiaryState) -> Dict[str, Any]:
    """Node 1: Standard extraction step mimicking image feature harvesting."""
    print("\n[Node 1: Mapping Extraction] Scraping metadata layers...")
    paths = state.get("image_paths", [])
    mappings = []

    # Simple simulated mapping output for demonstration tracking
    for idx, path in enumerate(paths):
        mappings.append({
            "scene_id": f"Scene {idx+1}",
            "file_path": path,
            "caption": f"Simulated descriptive tracking baseline for image file {path}"
        })
    return {"scene_mappings": mappings}

def vector_evaluation_node(state: RoutedDiaryState) -> Dict[str, Any]:
    """
    Node 2: Evaluates structural context confidence scores.
    In real workflows, this scans a vector database (e.g., Chroma, Pinecone).
    """
    print("\n[Node 2: Vector Evaluation] Checking data integrity metrics...")
    mappings = state.get("scene_mappings", [])

    # Safeguard check: If zero valid files were scraped, drop confidence to zero
    if not mappings or any("Error" in m["caption"] for m in mappings):
        confidence = 0.25
    else:
        # High mock confidence default score for optimal parameters
        confidence = 0.88

    print(f"📊 Evaluated Vector Database Confidence Score: {confidence}")
    return {"vector_confidence_score": confidence}

def diary_generation_node(state: RoutedDiaryState) -> Dict[str, Any]:
    """Node 3A: Main text generation engine node (Passed if confidence >= threshold)."""
    print("\n[Node 3A: Diary Generation] Threshold met! Invoking SmolLM2 narrative synthesis...")
    # Executing processing workflows ...
    return {"diary_narrative": "Dear Diary, Today the visual analysis layers successfully matched parameters..."}

def summary_generation_node(state: RoutedDiaryState) -> Dict[str, Any]:
    """Node 4: Processes downstream final analytics summary."""
    print("\n[Node 4: Summary Generation] Compiling downstream analysis report...")
    return {"emotional_summary": "System processed records within acceptable boundaries."}

def low_confidence_fallback_node(state: RoutedDiaryState) -> Dict[str, Any]:
    """Node 3B: Fallback router drop target. Completely avoids triggering heavy LLMs."""
    print("\n[Node 3B: Low Confidence Fallback] ⚠️ Bypassing LLM Engine to conserve resources!")
    return {
        "diary_narrative": "SYSTEM ALERT: Generation halted. The inputs yielded lower than 70% matching confidence.",
        "emotional_summary": "CRITICAL ERROR: Processing skipped due to poor metric alignment.",
        "metadata": {"abort_reason": "Low baseline confidence threshold constraint failure"}
    }

# ==========================================
# 3. THE CONDITIONAL ROUTING LOGIC EDGE
# ==========================================
def confidence_router_edge(state: RoutedDiaryState) -> Literal["continue_to_llm", "skip_to_fallback"]:
    """
    Acts as the conditional decider logic tracking metrics.
    Returns strings matching registered branch labels.
    """
    score = state.get("vector_confidence_score", 0.0)
    CONFIDENCE_THRESHOLD = 0.70 # Set matching threshold parameters

    if score >= CONFIDENCE_THRESHOLD:
        print("🔀 [Router Decision]: Confidence is HIGH. Routing to LLM Core nodes.")
        return "continue_to_llm"
    else:
        print("🔀 [Router Decision]: Confidence is LOW. Routing straight to Fast Fallback Node.")
        return "skip_to_fallback"

# ==========================================
# 4. ARCHITECTURE COMPILATION
# ==========================================
workflow = StateGraph(RoutedDiaryState)

# Register workflow operational nodes
workflow.add_node("extract_mappings", extraction_and_mapping_node)
workflow.add_node("evaluate_vector", vector_evaluation_node)
workflow.add_node("generate_diary", diary_generation_node)
workflow.add_node("generate_summary", summary_generation_node)
workflow.add_node("low_confidence_fallback", low_confidence_fallback_node)

# Construct static entry tracks
workflow.set_entry_point("extract_mappings")
workflow.add_edge("extract_mappings", "evaluate_vector")

# Bind conditional router tracking definitions
workflow.add_conditional_edges(
    "evaluate_vector",          # Node executing the router check directly after evaluation
    confidence_router_edge,     # The routing function evaluator logic
    {
        "continue_to_llm": "generate_diary",            # Path option A
        "skip_to_fallback": "low_confidence_fallback"   # Path option B
    }
)

# Connect downstream node terminal endings
workflow.add_edge("generate_diary", "generate_summary")
workflow.add_edge("generate_summary", END)
workflow.add_edge("low_confidence_fallback", END)

# Compile Graph
app = workflow.compile()


In [ ]:
# Test Case 1: Simulating High Matching Parameters
print("--- RUN 1: TESTING HIGH CONFIDENCE PIPELINE PATH ---")
high_conf_inputs = {"image_paths": ["img1.jpg", "img2.jpg"], "vector_confidence_score": 0.0}
output_success = app.invoke(high_conf_inputs)

print("\n" + "="*40 + "\n")

# Test Case 2: Forcing low matching arrays to trigger the bypass
print("--- RUN 2: TESTING REJECTION BYPASS PATH ---")
# Passing an error context indicator to mock sub-par matching conditions
low_conf_inputs = {"image_paths": [], "vector_confidence_score": 0.0}
output_bypassed = app.invoke(low_conf_inputs)
